# Advanced Problems with Solutions: Classes Are Callable

This notebook expands **Classes Are Callable** into a progressive, executable problem set with solutions, assertions, edge cases, and object-model best practices.

## You will practice

- class objects vs instance objects;
- `callable()`, `type()`, and `isinstance()`;
- class and instance namespaces;
- attribute shadowing;
- `__dict__` and `__slots__`;
- callable instances via `__call__`;
- `__new__` vs `__init__`;
- immutable subclasses;
- alternate constructors;
- `type` as a class factory;
- metaclass `__call__`;
- instance caching and weak references;
- registries, factories, and providers;
- bound methods and descriptors;
- practical design trade-offs.

> **Best practice:** This notebook deliberately inspects dunder attributes because the topic is Python's object model. In ordinary application code, prefer stable public APIs and built-ins unless low-level inspection is actually needed.

## 0. Baseline — a class object is callable

A class definition creates a **class object**. For an ordinary user-defined class, calling that class object creates and initializes an instance.

In [1]:
class Program:
    language = "Python"

    def __init__(self, name):
        self.name = name

    def describe(self):
        return f"{self.name} uses {self.language}"


print("callable(Program):", callable(Program))
p = Program("Demo")
print("type(p):", type(p))
print("isinstance(p, Program):", isinstance(p, Program))
print("p.describe():", p.describe())

assert callable(Program)
assert type(p) is Program
assert isinstance(p, Program)

callable(Program): True
type(p): <class '__main__.Program'>
isinstance(p, Program): True
p.describe(): Demo uses Python


# Problem 1 — Class namespace vs instance namespace

Create a `Server` with shared class attribute `protocol = "HTTPS"` and per-instance `host` and `port`.

Verify that two instances have independent state while both can read the class attribute.

In [2]:
# Solution 1
class Server:
    protocol = "HTTPS"

    def __init__(self, host, port):
        self.host = host
        self.port = port


s1 = Server("api.example.com", 443)
s2 = Server("admin.example.com", 8443)

print("s1.__dict__:", s1.__dict__)
print("s2.__dict__:", s2.__dict__)
print("Server.protocol:", Server.protocol)
print("s1.protocol:", s1.protocol)
print("s2.protocol:", s2.protocol)

s1.port = 9443

assert s1.port == 9443
assert s2.port == 8443
assert "protocol" not in s1.__dict__
assert "protocol" not in s2.__dict__
assert s1.protocol == "HTTPS"
assert s2.protocol == "HTTPS"

s1.__dict__: {'host': 'api.example.com', 'port': 443}
s2.__dict__: {'host': 'admin.example.com', 'port': 8443}
Server.protocol: HTTPS
s1.protocol: HTTPS
s2.protocol: HTTPS


### Why it works

`host` and `port` are written onto each instance. `protocol` lives in the class namespace, but normal attribute lookup lets instances read it.

# Problem 2 — Attribute shadowing

Start with `Theme.color = "blue"`. Assign `a.color = "red"`, then change `Theme.color` to `"green"`.

Predict which value `a` and `b` see.

In [3]:
# Solution 2
class Theme:
    color = "blue"


a = Theme()
b = Theme()
a.color = "red"

print("a.__dict__:", a.__dict__)
print("b.__dict__:", b.__dict__)

Theme.color = "green"

print("a.color:", a.color)
print("b.color:", b.color)

assert a.color == "red"
assert b.color == "green"

a.__dict__: {'color': 'red'}
b.__dict__: {}
a.color: red
b.color: green


`a.color` shadows the class attribute because an instance attribute with that name exists. `b` has no instance-level `color`, so lookup reaches the class.

# Problem 3 — Delete a shadowing instance attribute

Predict the final value of `x.mode` after deleting the instance attribute.

In [4]:
# Solution 3
class Config:
    mode = "class-default"


x = Config()
x.mode = "instance-value"
assert x.mode == "instance-value"

del x.mode

print("x.__dict__:", x.__dict__)
print("x.mode:", x.mode)

assert x.__dict__ == {}
assert x.mode == "class-default"

x.__dict__: {}
x.mode: class-default


# Problem 4 — `type(obj)` vs a misleading `obj.__class__`

Demonstrate why `type(obj)` is the reliable exact runtime type check.

In [5]:
# Solution 4
class Disguised:
    __class__ = str


d = Disguised()

print("type(d):", type(d))
print("d.__class__:", d.__class__)

assert type(d) is Disguised
assert d.__class__ is str

type(d): <class '__main__.Disguised'>
d.__class__: <class 'str'>


### Best practice

Use `type(obj)` for exact runtime type introspection. In polymorphic code, usually prefer `isinstance(obj, SomeClass)` because it respects inheritance.

# Problem 5 — Exact type checks vs inheritance-aware checks

Create `Animal` and subclass `Dog`. Compare `type(...) is ...` with `isinstance(...)`.

In [6]:
# Solution 5
class Animal:
    pass


class Dog(Animal):
    pass


pet = Dog()

checks = {
    "type(pet) is Animal": type(pet) is Animal,
    "type(pet) is Dog": type(pet) is Dog,
    "isinstance(pet, Animal)": isinstance(pet, Animal),
    "isinstance(pet, Dog)": isinstance(pet, Dog),
}

for label, result in checks.items():
    print(f"{label}: {result}")

assert type(pet) is not Animal
assert type(pet) is Dog
assert isinstance(pet, Animal)
assert isinstance(pet, Dog)

type(pet) is Animal: False
type(pet) is Dog: True
isinstance(pet, Animal): True
isinstance(pet, Dog): True


# Problem 6 — What exactly is callable?

Predict `callable(...)` for a class, function, lambda, integer, plain instance, and instance implementing `__call__`.

In [7]:
# Solution 6
def regular_function():
    return "called"


class Plain:
    pass


class Multiplier:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, value):
        return self.factor * value


objects = {
    "class Plain": Plain,
    "function": regular_function,
    "lambda": lambda x: x + 1,
    "integer": 42,
    "Plain instance": Plain(),
    "Multiplier instance": Multiplier(3),
}

for name, obj in objects.items():
    print(f"{name:22} callable={callable(obj)}")

triple = objects["Multiplier instance"]
assert callable(Plain)
assert callable(regular_function)
assert not callable(42)
assert not callable(objects["Plain instance"])
assert callable(triple)
assert triple(10) == 30

class Plain            callable=True
function               callable=True
lambda                 callable=True
integer                callable=False
Plain instance         callable=False
Multiplier instance    callable=True


# Problem 7 — Stateful callable instance

Implement `CallCounter` so an instance behaves like a function while preserving call state.

In [8]:
# Solution 7
class CallCounter:
    def __init__(self):
        self.count = 0

    def __call__(self, value):
        self.count += 1
        return f"Call #{self.count}: {value}"


counter = CallCounter()
print(counter("alpha"))
print(counter("beta"))
print(counter("gamma"))

assert callable(counter)
assert counter.count == 3

Call #1: alpha
Call #2: beta
Call #3: gamma


Callable instances are useful when function-like behavior needs persistent configuration, state, caching, or instrumentation.

# Problem 8 — Callable range validator

Create `RangeValidator(min_value, max_value)` with inclusive boundaries and constructor validation.

In [9]:
# Solution 8
class RangeValidator:
    def __init__(self, min_value, max_value):
        if min_value > max_value:
            raise ValueError("min_value cannot exceed max_value")
        self.min_value = min_value
        self.max_value = max_value

    def __call__(self, value):
        return self.min_value <= value <= self.max_value


validator = RangeValidator(10, 20)

for value in (5, 10, 15, 20, 25):
    print(value, "->", validator(value))

assert validator(5) is False
assert validator(10) is True
assert validator(15) is True
assert validator(20) is True
assert validator(25) is False

try:
    RangeValidator(20, 10)
except ValueError as exc:
    print("Expected error:", exc)
else:
    raise AssertionError("Expected ValueError")

5 -> False
10 -> True
15 -> True
20 -> True
25 -> False
Expected error: min_value cannot exceed max_value


# Problem 9 — Trace `__new__` and `__init__`

Instrument a class to observe normal construction order.

In [10]:
# Solution 9
class TracedObject:
    def __new__(cls, value):
        print(f"1. __new__: cls={cls.__name__}, value={value!r}")
        obj = super().__new__(cls)
        print(f"2. __new__: created id={id(obj)}")
        return obj

    def __init__(self, value):
        print(f"3. __init__: self id={id(self)}, value={value!r}")
        self.value = value
        print("4. __init__: finished")


item = TracedObject("example")
print("item.value:", item.value)

assert type(item) is TracedObject
assert item.value == "example"

1. __new__: cls=TracedObject, value='example'
2. __new__: created id=2515611415760
3. __init__: self id=2515611415760, value='example'
4. __init__: finished
item.value: example


`__new__` creates or returns the object. `__init__` initializes an already-created object.

# Problem 10 — `__init__` must return `None`

Deliberately return a non-`None` value from `__init__`, catch the resulting error, then correct it.

In [11]:
# Solution 10
class BrokenInitializer:
    def __init__(self):
        return "not allowed"


try:
    BrokenInitializer()
except TypeError as exc:
    print("Expected TypeError:", exc)


class CorrectInitializer:
    def __init__(self):
        self.ready = True


obj = CorrectInitializer()
assert obj.ready is True

Expected TypeError: __init__() should return None, not 'str'


# Problem 11 — `__new__` returns another type

Predict whether `__init__` runs when `__new__` returns a `dict` instead of an instance of the requested class.

In [12]:
# Solution 11
class Surprise:
    def __new__(cls):
        print("__new__ returning a dict")
        return {"created_by": cls.__name__}

    def __init__(self):
        print("__init__ ran")


result = Surprise()
print("result:", result)
print("type(result):", type(result))

assert type(result) is dict
assert result["created_by"] == "Surprise"
assert not isinstance(result, Surprise)

__new__ returning a dict
result: {'created_by': 'Surprise'}
type(result): <class 'dict'>


If `__new__` returns an unrelated object, normal initialization of the requested class does not continue with that class's `__init__`.

# Problem 12 — Immutable subclass with `__new__`

Subclass `tuple` to create `Point2D(x, y)`. Because tuples are immutable, create the underlying value in `__new__`.

In [13]:
# Solution 12
class Point2D(tuple):
    def __new__(cls, x, y):
        return super().__new__(cls, (x, y))

    @property
    def x(self):
        return self[0]

    @property
    def y(self):
        return self[1]


point = Point2D(3, 4)
print("point:", point)
print("x:", point.x)
print("y:", point.y)

assert tuple(point) == (3, 4)
assert point.x == 3
assert point.y == 4
assert isinstance(point, tuple)
assert isinstance(point, Point2D)

point: (3, 4)
x: 3
y: 4


# Problem 13 — Singleton mechanics with `__new__`

Return the same instance every time while preventing repeated `__init__` calls from resetting state.

> This is educational; global singletons can make testing and dependency management harder.

In [14]:
# Solution 13
class ApplicationState:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance._initialized = False
        return cls._instance

    def __init__(self):
        if self._initialized:
            return
        self.settings = {}
        self._initialized = True


state_a = ApplicationState()
state_a.settings["theme"] = "dark"
state_b = ApplicationState()

print("state_a is state_b:", state_a is state_b)
print("settings:", state_b.settings)

assert state_a is state_b
assert state_b.settings["theme"] == "dark"

state_a is state_b: True
settings: {'theme': 'dark'}


# Problem 14 — Factory-like `__new__`

Create `Number(text)` that returns an `int` or `float` instead of a `Number` instance.

In [15]:
# Solution 14
class Number:
    def __new__(cls, text):
        if not isinstance(text, str):
            raise TypeError("text must be a string")

        try:
            if any(marker in text.lower() for marker in (".", "e")):
                return float(text)
            return int(text)
        except ValueError as exc:
            raise ValueError(f"Cannot parse {text!r} as a number") from exc


whole = Number("42")
fractional = Number("3.14")
scientific = Number("1e3")

print(whole, type(whole))
print(fractional, type(fractional))
print(scientific, type(scientific))

assert whole == 42 and type(whole) is int
assert fractional == 3.14 and type(fractional) is float
assert scientific == 1000.0 and type(scientific) is float

42 <class 'int'>
3.14 <class 'float'>
1000.0 <class 'float'>


### Design note

This is legal but surprising: calling a class usually implies receiving an instance of that class. A named factory function such as `parse_number(...)` is often clearer.

# Problem 15 — Alternate constructor with `@classmethod`

Implement `User.from_full_name("Ada Lovelace")` using `cls(...)`, not `User(...)`.

In [16]:
# Solution 15
class User:
    def __init__(self, first_name, last_name):
        self.first_name = first_name
        self.last_name = last_name

    @classmethod
    def from_full_name(cls, full_name):
        parts = full_name.split()
        if len(parts) != 2:
            raise ValueError("Expected exactly two name components")
        first_name, last_name = parts
        return cls(first_name, last_name)

    def __repr__(self):
        return f"{type(self).__name__}({self.first_name!r}, {self.last_name!r})"


user = User.from_full_name("Ada Lovelace")
print(user)

assert user.first_name == "Ada"
assert user.last_name == "Lovelace"

User('Ada', 'Lovelace')


# Problem 16 — Alternate constructor preserves subclasses

Prove that inherited `from_full_name` constructs the subclass when it uses `cls(...)`.

In [17]:
# Solution 16
class AdminUser(User):
    role = "admin"


admin = AdminUser.from_full_name("Grace Hopper")
print(admin)
print("type(admin):", type(admin))
print("role:", admin.role)

assert type(admin) is AdminUser
assert isinstance(admin, User)
assert admin.role == "admin"

AdminUser('Grace', 'Hopper')
type(admin): <class '__main__.AdminUser'>
role: admin


# Problem 17 — `__dict__` is not guaranteed

Use `__slots__` to make instances without a normal instance dictionary.

In [18]:
# Solution 17
class Coordinate:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y


coord = Coordinate(10, 20)
print("coord.x:", coord.x)
print("coord.y:", coord.y)
print("has __dict__:", hasattr(coord, "__dict__"))

try:
    coord.z = 30
except AttributeError as exc:
    print("Expected AttributeError:", exc)

assert not hasattr(coord, "__dict__")
assert coord.x == 10
assert coord.y == 20

coord.x: 10
coord.y: 20
has __dict__: False
Expected AttributeError: 'Coordinate' object has no attribute 'z' and no __dict__ for setting new attributes


# Problem 18 — Callable strategy object

Create a configurable `Discount(percent)` object that becomes callable on prices.

In [19]:
# Solution 18
class Discount:
    def __init__(self, percent):
        if not 0 <= percent <= 100:
            raise ValueError("percent must be between 0 and 100")
        self.percent = percent

    def __call__(self, price):
        if price < 0:
            raise ValueError("price must be non-negative")
        return price * (1 - self.percent / 100)


no_discount = Discount(0)
ten_percent = Discount(10)
free = Discount(100)

print("10% off 250 ->", ten_percent(250))

assert no_discount(250) == 250
assert ten_percent(250) == 225
assert free(250) == 0

10% off 250 -> 225.0


# Problem 19 — A class is usually an instance of `type`

Inspect the relationship between a class, its metaclass, and an instance.

In [20]:
# Solution 19
class Sample:
    pass


sample = Sample()

print("type(Sample):", type(Sample))
print("isinstance(Sample, type):", isinstance(Sample, type))
print("type(sample):", type(sample))

assert type(Sample) is type
assert isinstance(Sample, type)
assert type(sample) is Sample

type(Sample): <class 'type'>
isinstance(Sample, type): True
type(sample): <class '__main__.Sample'>


Mental model:

```text
Sample   --instance of--> type
sample   --instance of--> Sample
```

# Problem 20 — Dynamically create a class with `type`

Use the three-argument form `type(name, bases, namespace)` to construct a class dynamically.

In [21]:
# Solution 20
def greet(self, name):
    return f"{self.greeting}, {name}!"


DynamicGreeter = type(
    "DynamicGreeter",
    (),
    {
        "greeting": "Hello",
        "greet": greet,
    },
)

dynamic = DynamicGreeter()

print("DynamicGreeter:", DynamicGreeter)
print("type(DynamicGreeter):", type(DynamicGreeter))
print(dynamic.greet("Python"))

assert dynamic.greet("Python") == "Hello, Python!"
assert type(DynamicGreeter) is type

DynamicGreeter: <class '__main__.DynamicGreeter'>
type(DynamicGreeter): <class 'type'>
Hello, Python!


# Problem 21 — Intercept class calls with a metaclass

Create a metaclass that logs class calls before delegating to `type.__call__`.

In [22]:
# Solution 21
class LoggingMeta(type):
    def __call__(cls, *args, **kwargs):
        print(f"Calling class: {cls.__name__}")
        print(f"args={args}")
        print(f"kwargs={kwargs}")
        result = super().__call__(*args, **kwargs)
        print(f"Created: {result!r}")
        return result


class Task(metaclass=LoggingMeta):
    def __init__(self, title, *, priority="normal"):
        self.title = title
        self.priority = priority

    def __repr__(self):
        return f"Task(title={self.title!r}, priority={self.priority!r})"


task = Task("Ship release", priority="high")
assert isinstance(task, Task)
assert task.priority == "high"

Calling class: Task
args=('Ship release',)
kwargs={'priority': 'high'}
Created: Task(title='Ship release', priority='high')


For ordinary classes, metaclass `__call__` is the class-call hook that normally coordinates `__new__` and `__init__`.

# Problem 22 — Validate constructed objects in a metaclass

Create `PositiveIdMeta` that validates a positive integer `id` after ordinary construction succeeds.

In [23]:
# Solution 22
class PositiveIdMeta(type):
    def __call__(cls, *args, **kwargs):
        obj = super().__call__(*args, **kwargs)

        if not isinstance(obj.id, int) or isinstance(obj.id, bool):
            raise TypeError("id must be an integer")
        if obj.id <= 0:
            raise ValueError("id must be positive")
        return obj


class Entity(metaclass=PositiveIdMeta):
    def __init__(self, entity_id, name):
        self.id = entity_id
        self.name = name


good = Entity(1, "alpha")
assert good.id == 1

for invalid in (0, -5):
    try:
        Entity(invalid, "bad")
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError("Expected ValueError")

for invalid in ("1", 1.5, True):
    try:
        Entity(invalid, "bad")
    except TypeError as exc:
        print("Expected TypeError:", exc)
    else:
        raise AssertionError("Expected TypeError")

Expected ValueError: id must be positive
Expected ValueError: id must be positive
Expected TypeError: id must be an integer
Expected TypeError: id must be an integer
Expected TypeError: id must be an integer


### Best practice

Metaclasses are powerful but expensive in conceptual complexity. Ordinary constructor validation is usually simpler unless the behavior genuinely belongs at class/metaclass level.

# Problem 23 — Cache instances in a metaclass

Return the same object for repeated calls with identical hashable constructor arguments.

In [24]:
# Solution 23
class CachedMeta(type):
    _instances = {}

    def __call__(cls, *args, **kwargs):
        key = (cls, args, tuple(sorted(kwargs.items())))
        if key not in cls._instances:
            cls._instances[key] = super().__call__(*args, **kwargs)
        return cls._instances[key]


class Service(metaclass=CachedMeta):
    def __init__(self, name, *, region="eu"):
        self.name = name
        self.region = region


a = Service("payments")
b = Service("payments")
c = Service("search")
d = Service("payments", region="us")
e = Service("payments", region="us")

print("a is b:", a is b)
print("a is c:", a is c)
print("d is e:", d is e)

assert a is b
assert a is not c
assert d is e
assert a is not d

a is b: True
a is c: False
d is e: True


A production cache also needs policy decisions about lifetime, eviction, thread safety, unhashable arguments, and constructor side effects.

# Problem 24 — Weak-reference cache

Use `weakref.WeakValueDictionary` so the cache does not necessarily keep instances alive forever.

In [25]:
# Solution 24
import gc
import weakref


class WeakCachedMeta(type):
    _instances = weakref.WeakValueDictionary()

    def __call__(cls, *args, **kwargs):
        key = (cls, args, tuple(sorted(kwargs.items())))
        instance = cls._instances.get(key)
        if instance is None:
            instance = super().__call__(*args, **kwargs)
            cls._instances[key] = instance
        return instance


class ConnectionProfile(metaclass=WeakCachedMeta):
    def __init__(self, host, port):
        self.host = host
        self.port = port


p1 = ConnectionProfile("db.example.com", 5432)
p2 = ConnectionProfile("db.example.com", 5432)

print("p1 is p2:", p1 is p2)
print("cache size while referenced:", len(WeakCachedMeta._instances))
assert p1 is p2

del p1
del p2
gc.collect()
print("cache size after deleting strong refs:", len(WeakCachedMeta._instances))

p1 is p2: True
cache size while referenced: 1
cache size after deleting strong refs: 0


# Problem 25 — Class registry and factory

Store classes in a registry, then instantiate them by calling the selected class.

In [26]:
# Solution 25
class PluginRegistry:
    def __init__(self):
        self._classes = {}

    def register(self, name, cls):
        if not isinstance(cls, type):
            raise TypeError("Only classes may be registered")
        if name in self._classes:
            raise ValueError(f"Plugin {name!r} is already registered")
        self._classes[name] = cls

    def create(self, name, *args, **kwargs):
        try:
            cls = self._classes[name]
        except KeyError as exc:
            raise KeyError(f"Unknown plugin: {name!r}") from exc
        return cls(*args, **kwargs)

    def names(self):
        return tuple(sorted(self._classes))


class JsonExporter:
    def __init__(self, indent=2):
        self.indent = indent

    def export(self, value):
        import json
        return json.dumps(value, indent=self.indent)


class ReprExporter:
    def export(self, value):
        return repr(value)


registry = PluginRegistry()
registry.register("json", JsonExporter)
registry.register("repr", ReprExporter)

json_exporter = registry.create("json", indent=4)
repr_exporter = registry.create("repr")

print("registered:", registry.names())
print(json_exporter.export({"x": 1}))
print(repr_exporter.export({"x": 1}))

assert registry.names() == ("json", "repr")
assert isinstance(json_exporter, JsonExporter)
assert isinstance(repr_exporter, ReprExporter)

registered: ('json', 'repr')
{
    "x": 1
}
{'x': 1}


# Problem 26 — Accept any callable factory

Implement `build(factory, *args, **kwargs)` that supports a class, function, or callable instance.

In [27]:
# Solution 26
def build(factory, *args, **kwargs):
    if not callable(factory):
        raise TypeError(f"factory must be callable, got {type(factory).__name__}")
    return factory(*args, **kwargs)


class Product:
    def __init__(self, name):
        self.name = name


def make_upper(text):
    return text.upper()


class Prefixer:
    def __init__(self, prefix):
        self.prefix = prefix

    def __call__(self, text):
        return f"{self.prefix}{text}"


product = build(Product, "keyboard")
upper = build(make_upper, "python")
identifier = build(Prefixer("ID-"), "42")

print(product.name)
print(upper)
print(identifier)

assert isinstance(product, Product)
assert upper == "PYTHON"
assert identifier == "ID-42"

try:
    build(123, "ignored")
except TypeError as exc:
    print("Expected error:", exc)
else:
    raise AssertionError("Expected TypeError")

keyboard
PYTHON
ID-42
Expected error: factory must be callable, got int


# Problem 27 — Inspect callable signatures

Use `inspect.signature` on a function, class, and callable instance.

In [28]:
# Solution 27
import inspect


def transform(value, *, uppercase=False):
    return value.upper() if uppercase else value


class Record:
    def __init__(self, key, value=None):
        self.key = key
        self.value = value


class Formatter:
    def __call__(self, value, width=10):
        return f"{value:>{width}}"


formatter = Formatter()

print("function:", inspect.signature(transform))
print("class:", inspect.signature(Record))
print("callable instance:", inspect.signature(formatter))

function: (value, *, uppercase=False)
class: (key, value=None)
callable instance: (value, width=10)


Clear signatures help readers, IDEs, tests, dependency-injection systems, and frameworks understand how a callable should be used.

# Problem 28 — Debug a mutable class attribute bug

Reproduce shared mutable state, then fix it by moving the list into `__init__`.

In [29]:
# Solution 28
class BuggyShoppingCart:
    items = []

    def add(self, item):
        self.items.append(item)


buggy_a = BuggyShoppingCart()
buggy_b = BuggyShoppingCart()
buggy_a.add("book")

print("buggy_a.items:", buggy_a.items)
print("buggy_b.items:", buggy_b.items)
assert buggy_b.items == ["book"]


class ShoppingCart:
    def __init__(self):
        self.items = []

    def add(self, item):
        self.items.append(item)


cart_a = ShoppingCart()
cart_b = ShoppingCart()
cart_a.add("book")

print("cart_a.items:", cart_a.items)
print("cart_b.items:", cart_b.items)

assert cart_a.items == ["book"]
assert cart_b.items == []
assert cart_a.items is not cart_b.items

buggy_a.items: ['book']
buggy_b.items: ['book']
cart_a.items: ['book']
cart_b.items: []


# Problem 29 — Raw function vs bound method

Inspect the same method through the class dictionary, the class, and an instance.

In [30]:
# Solution 29
class Calculator:
    def add(self, x, y):
        return x + y


calc = Calculator()
raw_function = Calculator.__dict__["add"]
class_access = Calculator.add
instance_access = calc.add

print("raw:", raw_function, type(raw_function))
print("class access:", class_access, type(class_access))
print("instance access:", instance_access, type(instance_access))

manual = raw_function(calc, 2, 3)
bound = calc.add(2, 3)

print("manual:", manual)
print("bound:", bound)

assert callable(raw_function)
assert callable(class_access)
assert callable(instance_access)
assert manual == 5
assert bound == 5

raw: <function Calculator.add at 0x00000249B629EB60> <class 'function'>
class access: <function Calculator.add at 0x00000249B629EB60> <class 'function'>
instance access: <bound method Calculator.add of <__main__.Calculator object at 0x00000249B62F0EC0>> <class 'method'>
manual: 5
bound: 5


Functions stored on classes participate in the descriptor protocol. Access through an instance can produce a bound method carrying the instance automatically.

# Problem 30 — Capstone: callable processing pipeline

Build `Pipeline(*steps)` where every step is callable and output flows from one step into the next.

In [31]:
# Solution 30
class Pipeline:
    def __init__(self, *steps):
        for index, step in enumerate(steps):
            if not callable(step):
                raise TypeError(
                    f"Step {index} must be callable; got {type(step).__name__}"
                )
        self.steps = tuple(steps)

    def __call__(self, value):
        for step in self.steps:
            value = step(value)
        return value

    def __repr__(self):
        names = [
            getattr(step, "__name__", type(step).__name__)
            for step in self.steps
        ]
        return f"Pipeline({', '.join(names)})"


def strip_text(text):
    return text.strip()


class Lowercase:
    def __call__(self, text):
        return text.lower()


class ReplaceSpaces:
    def __init__(self, replacement):
        self.replacement = replacement

    def __call__(self, text):
        return text.replace(" ", self.replacement)


pipeline = Pipeline(strip_text, Lowercase(), ReplaceSpaces("-"))
source = "  Hello Python World  "
result = pipeline(source)

print("pipeline:", pipeline)
print("source:", repr(source))
print("result:", repr(result))

assert callable(pipeline)
assert result == "hello-python-world"

try:
    Pipeline(strip_text, 123)
except TypeError as exc:
    print("Expected error:", exc)
else:
    raise AssertionError("Expected TypeError")

pipeline: Pipeline(strip_text, Lowercase, ReplaceSpaces)
source: '  Hello Python World  '
result: 'hello-python-world'
Expected error: Step 1 must be callable; got int


# Problem 31 — Count successful initializations with a class decorator

Wrap `__init__` so a counter increments only when initialization succeeds.

In [32]:
# Solution 31
from functools import wraps


def count_instances(cls):
    original_init = cls.__init__
    cls.instances_created = 0

    @wraps(original_init)
    def wrapped_init(self, *args, **kwargs):
        original_init(self, *args, **kwargs)
        cls.instances_created += 1

    cls.__init__ = wrapped_init
    return cls


@count_instances
class Message:
    def __init__(self, text):
        if not text:
            raise ValueError("text cannot be empty")
        self.text = text


Message("one")
Message("two")
assert Message.instances_created == 2

try:
    Message("")
except ValueError:
    pass

print("instances_created:", Message.instances_created)
assert Message.instances_created == 2

instances_created: 2


# Problem 32 — Count successful class calls with a metaclass

Move the counter to metaclass `__call__`, a more direct hook for class invocation.

In [33]:
# Solution 32
class CountingMeta(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        cls.instances_created = 0
        return cls

    def __call__(cls, *args, **kwargs):
        obj = super().__call__(*args, **kwargs)
        cls.instances_created += 1
        return obj


class Job(metaclass=CountingMeta):
    def __init__(self, name):
        if not name:
            raise ValueError("name cannot be empty")
        self.name = name


Job("build")
Job("test")
assert Job.instances_created == 2

try:
    Job("")
except ValueError:
    pass

print("successful constructions:", Job.instances_created)
assert Job.instances_created == 2

successful constructions: 2


# Problem 33 — Per-subclass metaclass caching

Ensure subclasses do not accidentally share cached instances.

In [34]:
# Solution 33
class PerClassCacheMeta(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        cls._instance_cache = {}
        return cls

    def __call__(cls, *args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))
        if key not in cls._instance_cache:
            cls._instance_cache[key] = super().__call__(*args, **kwargs)
        return cls._instance_cache[key]


class BaseClient(metaclass=PerClassCacheMeta):
    def __init__(self, environment):
        self.environment = environment


class ApiClient(BaseClient):
    pass


class AdminClient(BaseClient):
    pass


api_1 = ApiClient("prod")
api_2 = ApiClient("prod")
admin_1 = AdminClient("prod")
admin_2 = AdminClient("prod")

print("api_1 is api_2:", api_1 is api_2)
print("admin_1 is admin_2:", admin_1 is admin_2)
print("api_1 is admin_1:", api_1 is admin_1)

assert api_1 is api_2
assert admin_1 is admin_2
assert api_1 is not admin_1

api_1 is api_2: True
admin_1 is admin_2: True
api_1 is admin_1: False


# Problem 34 — Validate subclass APIs at class creation time

Require concrete subclasses to expose a callable `handle` method. A bad class should fail during class definition.

In [35]:
# Solution 34
class HandlerMeta(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        if name != "BaseHandler":
            handler = getattr(cls, "handle", None)
            if not callable(handler):
                raise TypeError(f"{name} must define a callable 'handle' method")
        return cls


class BaseHandler(metaclass=HandlerMeta):
    pass


class GoodHandler(BaseHandler):
    def handle(self, value):
        return value * 2


good = GoodHandler()
assert good.handle(5) == 10

try:
    class BadHandler(BaseHandler):
        handle = 42
except TypeError as exc:
    print("Expected class-definition error:", exc)
else:
    raise AssertionError("Expected TypeError")

Expected class-definition error: BadHandler must define a callable 'handle' method


# Problem 35 — Immutable normalized string

Subclass `str` and normalize the value inside `__new__`.

In [36]:
# Solution 35
class NormalizedString(str):
    def __new__(cls, value):
        if not isinstance(value, str):
            raise TypeError("value must be a string")
        normalized = value.strip().lower()
        return super().__new__(cls, normalized)


text = NormalizedString("   Hello PYTHON   ")
print(repr(text))
print(type(text))

assert text == "hello python"
assert isinstance(text, str)
assert isinstance(text, NormalizedString)

'hello python'
<class '__main__.NormalizedString'>


# Problem 36 — Provider objects

Create two callable dependency providers:

- `Provider`: fresh instance every call;
- `SingletonProvider`: at most one instance.

In [37]:
# Solution 36
class Provider:
    def __init__(self, cls, *args, **kwargs):
        if not isinstance(cls, type):
            raise TypeError("cls must be a class")
        self.cls = cls
        self.args = args
        self.kwargs = kwargs

    def __call__(self):
        return self.cls(*self.args, **self.kwargs)


class SingletonProvider(Provider):
    def __init__(self, cls, *args, **kwargs):
        super().__init__(cls, *args, **kwargs)
        self._instance = None

    def __call__(self):
        if self._instance is None:
            self._instance = super().__call__()
        return self._instance


class Repository:
    def __init__(self, name):
        self.name = name


fresh_provider = Provider(Repository, "users")
singleton_provider = SingletonProvider(Repository, "users")

fresh_a = fresh_provider()
fresh_b = fresh_provider()
single_a = singleton_provider()
single_b = singleton_provider()

print("fresh_a is fresh_b:", fresh_a is fresh_b)
print("single_a is single_b:", single_a is single_b)

assert fresh_a is not fresh_b
assert single_a is single_b

fresh_a is fresh_b: False
single_a is single_b: True


# Problem 37 — Predict class-call order

Predict the output before running the code.

In [38]:
# Solution 37
class Meta(type):
    def __call__(cls, *args, **kwargs):
        print("META CALL")
        return super().__call__(*args, **kwargs)


class A(metaclass=Meta):
    def __new__(cls):
        print("NEW")
        return super().__new__(cls)

    def __init__(self):
        print("INIT")


a = A()
assert isinstance(a, A)

META CALL
NEW
INIT


Expected order:

```text
META CALL
NEW
INIT
```

# Problem 38 — Make `__new__` return an integer

What changes if `__new__` returns `123`?

In [39]:
# Solution 38
class Meta2(type):
    def __call__(cls, *args, **kwargs):
        print("META CALL")
        return super().__call__(*args, **kwargs)


class B(metaclass=Meta2):
    def __new__(cls):
        print("NEW -> returning int")
        return 123

    def __init__(self):
        print("INIT SHOULD NOT RUN")


b = B()
print("b:", b)
print("type(b):", type(b))

assert b == 123
assert type(b) is int

META CALL
NEW -> returning int
b: 123
type(b): <class 'int'>


# Problem 39 — Callable command dispatcher

Create a dispatcher that accepts arbitrary callable command objects and invokes them by name.

In [40]:
# Solution 39
class CommandDispatcher:
    def __init__(self):
        self._commands = {}

    def register(self, name, command):
        if not callable(command):
            raise TypeError("command must be callable")
        if name in self._commands:
            raise ValueError(f"duplicate command: {name}")
        self._commands[name] = command

    def run(self, name, *args, **kwargs):
        try:
            command = self._commands[name]
        except KeyError as exc:
            raise KeyError(f"unknown command: {name}") from exc
        return command(*args, **kwargs)


class Add:
    def __call__(self, x, y):
        return x + y


class Power:
    def __init__(self, exponent):
        self.exponent = exponent

    def __call__(self, value):
        return value ** self.exponent


dispatcher = CommandDispatcher()
dispatcher.register("add", Add())
dispatcher.register("square", Power(2))

assert dispatcher.run("add", 2, 3) == 5
assert dispatcher.run("square", 9) == 81
print(dispatcher.run("add", 20, 22))
print(dispatcher.run("square", 12))

42
144


# Problem 40 — Typed class registry

Accept only subclasses of a required base class, then instantiate them by registered name.

In [41]:
# Solution 40
class BaseProcessor:
    def process(self, value):
        raise NotImplementedError


class TypedRegistry:
    def __init__(self, base_class):
        if not isinstance(base_class, type):
            raise TypeError("base_class must be a class")
        self.base_class = base_class
        self._types = {}

    def register(self, name, cls):
        if not isinstance(cls, type):
            raise TypeError("registered value must be a class")
        if not issubclass(cls, self.base_class):
            raise TypeError(
                f"{cls.__name__} must inherit from {self.base_class.__name__}"
            )
        self._types[name] = cls

    def create(self, name, *args, **kwargs):
        return self._types[name](*args, **kwargs)


class StripProcessor(BaseProcessor):
    def process(self, value):
        return value.strip()


class LowerProcessor(BaseProcessor):
    def process(self, value):
        return value.lower()


processors = TypedRegistry(BaseProcessor)
processors.register("strip", StripProcessor)
processors.register("lower", LowerProcessor)

stripper = processors.create("strip")
lowerer = processors.create("lower")

assert stripper.process(" x ") == "x"
assert lowerer.process("PyThOn") == "python"

try:
    processors.register("bad", dict)
except TypeError as exc:
    print("Expected type restriction:", exc)

Expected type restriction: dict must inherit from BaseProcessor


# Summary — best practices

1. Use `isinstance()` for most polymorphic type checks; use `type(x) is T` only for intentional exact-type logic.
2. Put per-object mutable state on instances, not shared class attributes.
3. Use `callable()` when an API accepts any callable rather than assuming the value is a function.
4. Use `__call__` for function-like objects that need persistent state or configuration.
5. Use `__new__` carefully; it is especially relevant for immutable subclasses or unusual construction control.
6. Keep `__init__` focused on initialization and let it return `None`.
7. Prefer `@classmethod` alternate constructors to surprising class-call behavior where possible.
8. Use metaclasses sparingly; factories, decorators, descriptors, and registries are often simpler.
9. Do not assume every object has an instance `__dict__`.
10. Remember that Python's callable protocol is broader than functions: classes, bound methods, and callable instances can all fit the same higher-level APIs.

# Additional unsolved advanced practice

### A. Retry policy
Create `RetryPolicy(max_attempts)` as a callable object that invokes another callable and retries after exceptions up to the configured limit.

### B. Construction timer
Create a metaclass whose `__call__` measures successful construction time and stores the elapsed duration on the returned instance.

### C. Bounded instance cache
Extend instance caching with a maximum cache size and a clear eviction policy.

### D. Constructor audit log
Create a metaclass that records class name, constructor arguments, success/failure, and exception type without swallowing exceptions.

### E. `__slots__` inheritance
Experiment with slotted bases and subclasses with and without `__slots__`. Inspect where new instance attributes can be stored.

### F. Metaclass conflict
Create two unrelated custom metaclasses, then multiple-inherit from classes using them. Explain the metaclass conflict and resolve it with a compatible metaclass.